# Auditoría de Estadísticas de Lesiones — TFG Guillermo Jiménez

Reproduce y explica, paso a paso, cómo se calcula cada una de las 9 métricas de lesión
que aparecen en la hoja **"4. Métricas Finales"** del archivo `Auditoria Lesiones TFG.xlsx`.

**Métricas calculadas (10):**
1. N Temporadas
2. Días última temporada
3. Media últimas 2 temp.
4. Media últimas 3 temp.
5. EWA Días (α=0.4)
6. Pendiente OLS
7. p-valor OLS
8. Sig. (α=10%)
9. Riesgo (low/medium/high)
10. Tipo de Tendencia (stable/worsening/improving)

In [ ]:
!pip install -q pandas numpy scipy openpyxl

## Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import os

## Rutas

In [ ]:
import os
DB = '/content'

AUDITORIA_PATH = os.path.join(DB, 'Auditoria Lesiones TFG.xlsx')

## Constantes del modelo

In [ ]:
ALPHA_EWA       = 0.4    # factor de decaimiento para la media exp. ponderada
UMBRAL_PVALOR   = 0.10   # umbral de significatividad estadística (10%)
UMBRAL_SLOPE    = 2.0    # días/año mínimos para no clasificar la tendencia como 'stable'
RIESGO_LOW      = 10     # días/temporada: por debajo → riesgo bajo
RIESGO_MEDIUM   = 40     # días/temporada: entre 10 y 40 → riesgo medio; >40 → alto

## Configuración del jugador a auditar

Edita la variable `JUGADOR` con el nombre del jugador que quieras auditar.
También puedes poner su ID numérico de Transfermarkt en `TM_ID`.
Pon `VALIDAR_TODOS = True` para validar todos los jugadores del Excel.

In [ ]:
JUGADOR      = "James Milner"  # Cambia el nombre
TM_ID        = None            # O pon el ID numérico de TM
VALIDAR_TODOS = False          # Cambia a True para validar todos

## Carga de datos

In [ ]:
def cargar_datos(path=AUDITORIA_PATH):
    """Lee las 4 hojas relevantes del Excel de auditoría."""
    xl = pd.ExcelFile(path)

    def buscar_hoja(patron):
        """Busca una hoja cuyo nombre contenga el patrón (sin tildes, minúsculas)."""
        import unicodedata
        def normalizar(s):
            return ''.join(
                c for c in unicodedata.normalize('NFD', s.lower())
                if unicodedata.category(c) != 'Mn'
            )
        patron_n = normalizar(patron)
        for nombre in xl.sheet_names:
            if patron_n in normalizar(nombre):
                return nombre
        raise ValueError(f"No se encontró ninguna hoja con '{patron}'. Hojas disponibles: {xl.sheet_names}")

    hoja_brutas   = buscar_hoja('Lesiones Brutas')
    hoja_serie    = buscar_hoja('Serie Temporal')
    hoja_metricas = buscar_hoja('Metricas Finales')

    brutas = xl.parse(hoja_brutas, header=1)
    brutas.columns = ['tm_id', 'jugador', 'equipo', 'temporada',
                      'tipo_lesion', 'fecha_inicio', 'fecha_fin',
                      'dias_baja', 'fila_resumen']

    serie = xl.parse(hoja_serie, header=1)
    serie.columns = ['tm_id', 'jugador', 'equipo', 'temporada',
                     'anio_temporada', 'dias_baja', 'pendiente_acum']

    metricas = xl.parse(hoja_metricas, header=1)
    metricas.columns = [
        'tm_id', 'jugador', 'equipo', 'n_temp',
        'dias_ult', 'media_ult2', 'media_ult3',
        'ewa_dias', 'pendiente', 'pvalor', 'p_menor_010',
        'sig_alfa10', 'riesgo', 'tipo_tendencia'
    ]

    return brutas, serie, metricas

## Lógica de cálculo de métricas

In [ ]:
def serie_temporal_jugador(df_serie, tm_id):
    """Filtra la serie temporal por jugador y devuelve lista [(año, días_baja)]."""
    datos = df_serie[df_serie['tm_id'] == tm_id].copy()
    datos = datos.dropna(subset=['anio_temporada', 'dias_baja'])
    datos['anio_temporada'] = datos['anio_temporada'].astype(int)
    datos = datos.sort_values('anio_temporada')
    return list(zip(datos['anio_temporada'].tolist(),
                    datos['dias_baja'].tolist()))


def calcular_metricas(serie, verbose=False, nombre=''):
    """
    Reproduce el cálculo de compute_injury_metrics() con cada paso documentado.
    Retorna dict con todas las métricas.
    """

    def sep(title=''):
        if verbose:
            print(f"\n  {'─'*55}")
            if title:
                print(f'  {title}')

    if verbose:
        print(f"\n{'═'*60}")
        print(f'  AUDITORÍA DE LESIONES — {nombre}')
        print(f"{'═'*60}")

    years = [s[0] for s in serie]
    days  = [s[1] for s in serie]
    n     = len(days)

    if verbose:
        sep('DATOS DE ENTRADA (Hoja: 2. Serie Temporal)')
        print(f'  Temporadas disponibles: {n}')
        for yr, d in serie:
            print(f'    {yr}/{str(yr+1)[2:]}  →  {d} días de baja')

    # MÉTRICA 1: N Temporadas
    sep('MÉTRICA 1 — N Temporadas')
    if verbose:
        print(f'  Fórmula : CONTAR filas en Serie Temporal para este jugador')
        print(f'  Resultado: N = {n}')

    # MÉTRICA 2: Días última temporada
    dias_ult = days[-1] if n >= 1 else 0
    sep('MÉTRICA 2 — Días última temporada')
    if verbose:
        print(f'  Temporada más reciente: {years[-1]}/{str(years[-1]+1)[2:]}')
        print(f'  Resultado: {dias_ult} días')

    # MÉTRICA 3: Media últimas 2 temporadas
    if n >= 2:
        vals2 = days[-2:]
        media2 = round(np.mean(vals2), 1)
        formula2 = f"({' + '.join(str(v) for v in vals2)}) / {len(vals2)}"
    else:
        vals2 = days
        media2 = round(float(days[-1]), 1)
        formula2 = f'{days[-1]} (solo 1 temporada disponible)'

    sep('MÉTRICA 3 — Media últimas 2 temporadas')
    if verbose:
        print(f'  Valores : {vals2}')
        print(f'  Cálculo : {formula2} = {media2}')

    # MÉTRICA 4: Media últimas 3 temporadas
    if n >= 3:
        vals3 = days[-3:]
        media3 = round(np.mean(vals3), 1)
    else:
        vals3 = days
        media3 = round(np.mean(days), 1)

    sep('MÉTRICA 4 — Media últimas 3 temporadas')
    if verbose:
        print(f'  Valores : {vals3}')
        print(f'  Resultado: {media3}')

    # MÉTRICA 5: EWA
    ewa = float(days[0])
    ewa_steps = [(years[0], days[0], ewa)]
    for yr, d in zip(years[1:], days[1:]):
        ewa = ALPHA_EWA * d + (1 - ALPHA_EWA) * ewa
        ewa_steps.append((yr, d, round(ewa, 3)))
    ewa_final = round(ewa, 1)

    sep(f'MÉTRICA 5 — EWA Días (α={ALPHA_EWA})')
    if verbose:
        print(f'  EWA_0 = {days[0]} días')
        for yr, d, e in ewa_steps[1:]:
            prev_e = ewa_steps[ewa_steps.index((yr, d, e)) - 1][2]
            print(f'  {yr}/{str(yr+1)[2:]} → {ALPHA_EWA}·{d} + {1-ALPHA_EWA}·{prev_e} = {e}')
        print(f'  EWA final = {ewa_final} días')

    # MÉTRICA 6+7: Regresión OLS
    if n >= 3:
        slope, intercept, r_value, pvalue, stderr = stats.linregress(years, days)
        slope  = round(slope, 3)
        pvalue = round(pvalue, 4)
        sep('MÉTRICA 6+7 — Pendiente OLS y p-valor')
        if verbose:
            print(f'  n = {n} temporadas')
            print(f'  β₁ (slope) = {slope} días/año')
            print(f'  p-valor    = {pvalue}')
    else:
        slope  = 0.0
        pvalue = 1.0
        sep('MÉTRICA 6+7 — Pendiente OLS y p-valor')
        if verbose:
            print(f'  Solo {n} temporada(s) — mínimo 3 para OLS.')
            print(f'  Valores por defecto: slope=0.0, p-valor=1.0')

    # MÉTRICA 8: Significatividad
    sig = pvalue < UMBRAL_PVALOR
    sep('MÉTRICA 8 — Sig. (α=10%)')
    if verbose:
        print(f'  p-valor = {pvalue}  → {"SÍ significativa" if sig else "NO significativa"}')

    # MÉTRICA 9: Riesgo
    media_general = round(np.mean(days), 1)
    if media_general < RIESGO_LOW:
        riesgo = 'low'
    elif media_general < RIESGO_MEDIUM:
        riesgo = 'medium'
    else:
        riesgo = 'high'

    sep('MÉTRICA 9 — Riesgo')
    if verbose:
        print(f'  Media general = {media_general} días/temporada')
        print(f'  Resultado: \'{riesgo}\'')

    # MÉTRICA 10: Tipo de Tendencia
    if abs(slope) < UMBRAL_SLOPE:
        tipo = 'stable'
    elif slope > 0:
        tipo = 'worsening'
    else:
        tipo = 'improving'

    sep('MÉTRICA 10 — Tipo de Tendencia')
    if verbose:
        print(f'  β₁ = {slope}  →  \'{tipo}\'')

    resultado = {
        'n_temp'      : n,
        'dias_ult'    : dias_ult,
        'media_ult2'  : media2,
        'media_ult3'  : media3,
        'ewa_dias'    : ewa_final,
        'pendiente'   : slope,
        'pvalor'      : pvalue,
        'p_menor_010' : 'SÍ' if sig else 'NO',
        'sig_alfa10'  : 'SÍ' if sig else 'NO',
        'riesgo'      : riesgo,
        'tipo_tend'   : tipo,
    }

    if verbose:
        sep()
        print(f"\n  {'─'*55}")
        print(f'  RESUMEN MÉTRICAS FINALES — {nombre}')
        print(f"  {'─'*55}")
        print(f"  {'N Temporadas':<30} {n}")
        print(f"  {'Días última temporada':<30} {dias_ult}")
        print(f"  {'Media últimas 2 temp.':<30} {media2}")
        print(f"  {'Media últimas 3 temp.':<30} {media3}")
        print(f"  {'EWA Días (α=0.4)':<30} {ewa_final}")
        print(f"  {'Pendiente OLS (días/año)':<30} {slope}")
        print(f"  {'p-valor OLS':<30} {pvalue}")
        print(f"  {'Significativa (α=10%)':<30} {'Sí' if sig else 'No'}")
        print(f"  {'Riesgo':<30} {riesgo}")
        print(f"  {'Tipo de Tendencia':<30} {tipo}")
        print(f"  {'─'*55}\n")

    return resultado

## Función: auditar un jugador concreto

In [ ]:
def auditar_jugador(nombre=None, tm_id=None, path=AUDITORIA_PATH):
    brutas, serie, metricas = cargar_datos(path)

    if tm_id is not None:
        fila = metricas[metricas['tm_id'] == tm_id]
        if fila.empty:
            fila = serie[serie['tm_id'] == tm_id]
            if not fila.empty:
                nombre = fila.iloc[0]['jugador']
                tm_id  = int(fila.iloc[0]['tm_id'])
    elif nombre is not None:
        fila = metricas[metricas['jugador'].str.contains(nombre, case=False, na=False)]
        if fila.empty:
            print(f'Jugador \'{nombre}\' no encontrado en Métricas Finales.')
            fila2 = serie[serie['jugador'].str.contains(nombre, case=False, na=False)]
            if fila2.empty:
                print('  Tampoco encontrado en Serie Temporal. Revisa el nombre.')
                return
            tm_id  = int(fila2.iloc[0]['tm_id'])
            nombre = fila2.iloc[0]['jugador']
        else:
            tm_id  = int(fila.iloc[0]['tm_id'])
            nombre = fila.iloc[0]['jugador']
    else:
        ejemplo = serie.iloc[0]
        tm_id   = int(ejemplo['tm_id'])
        nombre  = ejemplo['jugador']
        print(f'No se especificó jugador. Usando ejemplo: {nombre} (ID {tm_id})')

    datos_serie = serie_temporal_jugador(serie, tm_id)
    if not datos_serie:
        print(f'No hay datos de serie temporal para {nombre} (ID {tm_id})')
        return

    resultado = calcular_metricas(datos_serie, verbose=True, nombre=nombre)

    fila_excel = metricas[metricas['tm_id'] == tm_id]
    if not fila_excel.empty:
        r = fila_excel.iloc[0]
        print('  VALIDACIÓN vs Excel (Hoja 4. Métricas Finales):')
        print(f"  {'─'*55}")

        def check(campo, calc, excel_val, decimales=1):
            try:
                calc_r  = round(float(calc), decimales)
                excel_r = round(float(excel_val), decimales)
                ok = 'OK' if calc_r == excel_r else 'DIFF'
                print(f'  [{ok}] {campo:<28} Calc={calc_r:<8} Excel={excel_r}')
            except Exception:
                print(f'  [??] {campo:<28} Calc={calc}  Excel={excel_val} (no numérico)')

        check('N Temporadas',           resultado['n_temp'],    r['n_temp'],    0)
        check('Días última temp.',      resultado['dias_ult'],  r['dias_ult'],  0)
        check('Media últ. 2',           resultado['media_ult2'],r['media_ult2'])
        check('Media últ. 3',           resultado['media_ult3'],r['media_ult3'])
        check('EWA Días',               resultado['ewa_dias'],  r['ewa_dias'])
        check('Pendiente OLS',          resultado['pendiente'], r['pendiente'], 3)
        check('p-valor',                resultado['pvalor'],    r['pvalor'],    4)
        cat_ok = resultado['riesgo'] == str(r['riesgo']).strip().lower()
        print(f"  {'[OK]' if cat_ok else '[DIFF]'} {'Riesgo':<28} Calc={resultado['riesgo']:<8} Excel={r['riesgo']}")
        tipo_ok = resultado['tipo_tend'] == str(r['tipo_tendencia']).strip().lower()
        print(f"  {'[OK]' if tipo_ok else '[DIFF]'} {'Tipo Tendencia':<28} Calc={resultado['tipo_tend']:<8} Excel={r['tipo_tendencia']}")
        print(f"  {'─'*55}")

## Función: validar todos los jugadores

In [ ]:
def validar_todos(path=AUDITORIA_PATH, exportar=True):
    """Recalcula las métricas para todos los jugadores y compara con el Excel."""
    brutas, serie, metricas = cargar_datos(path)
    ids = metricas['tm_id'].dropna().unique()

    print(f'\nValidando {len(ids)} jugadores...\n')

    errores = []
    ok_total = 0

    for tm_id in ids:
        tm_id = int(tm_id)
        datos_serie = serie_temporal_jugador(serie, tm_id)
        if not datos_serie:
            continue

        calc = calcular_metricas(datos_serie, verbose=False)
        fila = metricas[metricas['tm_id'] == tm_id].iloc[0]
        nombre = str(fila['jugador'])

        diffs = {}
        campos_num = {
            'n_temp':    ('n_temp',    0),
            'dias_ult':  ('dias_ult',  0),
            'media_ult2':('media_ult2',1),
            'media_ult3':('media_ult3',1),
            'ewa_dias':  ('ewa_dias',  1),
            'pendiente': ('pendiente', 3),
            'pvalor':    ('pvalor',    4),
        }
        for c_key, (e_key, dec) in campos_num.items():
            try:
                c_val = round(float(calc[c_key]), dec)
                e_val = round(float(fila[e_key]), dec)
                if c_val != e_val:
                    diffs[c_key] = (c_val, e_val)
            except Exception:
                pass

        if calc['riesgo'] != str(fila['riesgo']).strip().lower():
            diffs['riesgo'] = (calc['riesgo'], fila['riesgo'])
        if calc['tipo_tend'] != str(fila['tipo_tendencia']).strip().lower():
            diffs['tipo_tend'] = (calc['tipo_tend'], fila['tipo_tendencia'])

        if diffs:
            errores.append({'tm_id': tm_id, 'jugador': nombre, 'diferencias': diffs})
        else:
            ok_total += 1

    total = len(ids)
    print(f'Coincidencias exactas : {ok_total}/{total}')
    print(f'Discrepancias         : {len(errores)}/{total}')

    if errores:
        print('\nPrimeros 10 con discrepancias:')
        for e in errores[:10]:
            print(f'  [{e["tm_id"]}] {e["jugador"]}')
            for campo, (calc_v, excel_v) in e['diferencias'].items():
                print(f'      {campo}: calculado={calc_v} | excel={excel_v}')

    if exportar and errores:
        rows = []
        for e in errores:
            for campo, (c, x) in e['diferencias'].items():
                rows.append({
                    'tm_id': e['tm_id'],
                    'jugador': e['jugador'],
                    'campo': campo,
                    'calculado': c,
                    'excel': x,
                    'diferencia': round(float(c) - float(x), 4)
                        if str(c).replace('.', '').replace('-', '').isdigit() else '—'
                })
        df_err = pd.DataFrame(rows)
        out = 'informe_validacion_lesiones.csv'
        df_err.to_csv(out, index=False, encoding='utf-8-sig')
        print(f'\nInforme exportado: {out}')

## Ejecución

Según el valor de `VALIDAR_TODOS` se audita un jugador concreto o se validan todos.

In [ ]:
if VALIDAR_TODOS:
    validar_todos(path=AUDITORIA_PATH)
else:
    auditar_jugador(nombre=JUGADOR, tm_id=TM_ID)

ValueError: Worksheet named '4. Métricas Finales' not found